<a href="https://colab.research.google.com/github/patrickrekyn/Analyse-de-donn-es-avec-R-gression-Lin-aire/blob/main/testquerry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate sentencepiece torch -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("spider")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [ ]:
def is_simple_query(example):

    query = example["query"].upper()

    forbidden_keywords = [
        "JOIN",
        "GROUP BY",
        "HAVING",
        "UNION",
        "INTERSECT",
        "EXCEPT",
        " EXISTS ",
        " IN ",
        "SELECT COUNT",
        "SELECT SUM",
        "SELECT AVG",
        "SELECT MAX",
        "SELECT MIN"
    ]

    for keyword in forbidden_keywords:

        if keyword in query:
            return False

    return True

In [ ]:
simple_dataset = dataset["train"].filter(is_simple_query)

Filter:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [ ]:
print(len(simple_dataset))

1787


In [ ]:
for i in range(5):

    print("\nQUESTION:")
    print(simple_dataset[i]["question"])

    print("\nSQL:")
    print(simple_dataset[i]["query"])

    print("\n-------------------")


QUESTION:
List the name, born state and age of the heads of departments ordered by age.

SQL:
SELECT name ,  born_state ,  age FROM head ORDER BY age

-------------------

QUESTION:
List the creation year, name and budget of each department.

SQL:
SELECT creation ,  name ,  budget_in_billions FROM department

-------------------

QUESTION:
What are the names of the heads who are born outside the California state?

SQL:
SELECT name FROM head WHERE born_state != 'California'

-------------------

QUESTION:
Which head's name has the substring 'Ha'? List the id and name.

SQL:
SELECT head_id ,  name FROM head WHERE name LIKE '%Ha%'

-------------------

QUESTION:
List the total number of horses on farms in ascending order.

SQL:
SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC

-------------------


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
MAX_LEN = 128

def preprocess_function(examples):

    inputs = [

        f"Generate a SIMPLE SQL query: {q}"

        for q in examples["question"]
    ]

    targets = examples["query"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_dataset = simple_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/1787 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
training_args = TrainingArguments(

    output_dir="./simple_sql_model",

    per_device_train_batch_size=4,

    num_train_epochs=5,

    learning_rate=3e-4,

    logging_steps=100,

    save_strategy="epoch",

    fp16=True
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset
)

In [ ]:
trainer.train()

Step,Training Loss
100,1.128964
200,0.279374
300,0.219787
400,0.192465
500,0.161978
600,0.154624
700,0.146004
800,0.131341
900,0.132015
1000,0.109212


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2235, training_loss=0.16710763830999933, metrics={'train_runtime': 250.0041, 'train_samples_per_second': 35.739, 'train_steps_per_second': 8.94, 'total_flos': 302319749038080.0, 'train_loss': 0.16710763830999933, 'epoch': 5.0})

In [ ]:
model.save_pretrained("simple_text2sql_model")

tokenizer.save_pretrained("simple_text2sql_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('simple_text2sql_model/tokenizer_config.json',
 'simple_text2sql_model/tokenizer.json')

In [ ]:
def generate_sql(question):

    input_text = f"""
    Generate ONLY a SIMPLE SQL query.

    Question:
    {question}

    SQL:
    """

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    )

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(

        **inputs,

        max_length=128,

        do_sample=False,

        num_beams=1,

        early_stopping=True
    )

    sql = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return sql

In [ ]:
def generate_sql(question):
    input_text = f"Generate ONLY a SIMPLE SQL query.\n\nQuestion:\n{question}\n\nSQL:\n"

    # On s'assure que les tensors vont sur le bon GPU dès la création
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100, # Préférable à max_length pour éviter de compter le prompt
        do_sample=False,
        num_beams=5,
        early_stopping=True
    )

    sql = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Nettoyage pour ne garder que la partie après "SQL:"
    if "SQL:" in sql:
        sql = sql.split("SQL:")[-1].strip()

    return sql

In [ ]:
print(generate_sql("Show all students"))

SELECT * FROM STUDENT


In [ ]:
print(generate_sql("Show all student names"))

SELECT Student_Name FROM Student


In [ ]:
print(generate_sql("Find students older than 18"))

SELECT Student FROM students WHERE Age > 18


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r simple_text2sql_model /content/drive/MyDrive/

In [ ]:
print(generate_sql("Show all student older than 30"))

SELECT DISTINCT student FROM STUDENT WHERE age > 30


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_path = "/content/drive/MyDrive/simple_text2sql_model"

tokenizer = T5Tokenizer.from_pretrained(model_path, local_files_only=True, legacy=False)

model = T5ForConditionalGeneration.from_pretrained(model_path, local_files_only=True)

In [ ]:
import os

# List the contents of the model_path directory
print(f"Contents of {model_path}:")
print(os.listdir(model_path))

Contents of /content/drive/MyDrive/simple_text2sql_model:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/simple_text2sql_model'

In [ ]:
import os

# List the contents of MyDrive to check for the model directory
print("Contents of /content/drive/MyDrive/")
print(os.listdir('/content/drive/MyDrive/'))

Contents of /content/drive/MyDrive/


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# List the contents of MyDrive to check for the model directory
print("Contents of /content/drive/MyDrive/")
print(os.listdir('/content/drive/MyDrive/'))

Contents of /content/drive/MyDrive/
['Colab Notebooks', 'RAZANADRAKOTO.pdf', 'insert client.sql', 'AGENCE.xlsx', 'RegressionLineaire.ipynb', 'simple_text2sql_model']


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_path = "/content/drive/MyDrive/simple_text2sql_model"

tokenizer = T5Tokenizer.from_pretrained(model_path)

model = T5ForConditionalGeneration.from_pretrained(model_path)

Loading weights:   0%|          | 0/131 [00:02<?, ?it/s]

In [ ]:
!pip install transformers sentencepiece -q

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-fr-en"

# Explicitly load tokenizer and model
tokenizer_translator = AutoTokenizer.from_pretrained(model_name)
model_translator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

translator = pipeline(
    "translation_fr_to_en",
    model=model_translator,
    tokenizer=tokenizer_translator
)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

KeyError: 'translation'

In [ ]:
french_sentence = "Comment allez-vous aujourd'hui?"

# Prepare the input for the model
inputs = tokenizer_translator(french_sentence, return_tensors="pt", truncation=True)

# Generate the translation
# Move inputs to the same device as the model (e.g., GPU if available)
# The model will automatically handle the translation generation
translated_tokens = model_translator.generate(
    **inputs.to(model_translator.device),
    max_new_tokens=128 # Limit the length of the generated translation
)

# Decode the generated tokens back to text
english_translation = tokenizer_translator.decode(translated_tokens[0], skip_special_tokens=True)

print(f"French: {french_sentence}")
print(f"English: {english_translation}")

French: Comment allez-vous aujourd'hui?
English: How are you today?


In [ ]:
from transformers import pipeline

translator = pipeline(
    task="translation_fr_to_en",
    model="Helsinki-NLP/opus-mt-fr-en"
)

KeyError: 'translation'

In [ ]:
!pip install transformers sentencepiece sacremoses -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 47.6 MB/s eta 0:00:00


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

In [ ]:
model_name = "Helsinki-NLP/opus-mt-fr-en"

tokenizer_trans = MarianTokenizer.from_pretrained(model_name)

model_trans = MarianMTModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

In [ ]:
def translate_fr_to_en(text):

    inputs = tokenizer_trans(
        text,
        return_tensors="pt",
        padding=True
    )

    translated = model_trans.generate(**inputs)

    result = tokenizer_trans.decode(
        translated[0],
        skip_special_tokens=True
    )

    return result

In [ ]:
print(
    translate_fr_to_en(
        "Afficher tous les étudiants"
    )
)

View all students


In [ ]:
def french_to_sql(question_fr):

    # Traduction FR → EN
    english_question = translate_fr_to_en(question_fr)

    print("English:", english_question)

    # Prompt SQL
    input_text = f"""
    Generate ONLY a SIMPLE SQL query.

    Question:
    {english_question}

    SQL:
    """

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_length=128,
        do_sample=False,
        num_beams=1
    )

    sql = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return sql

In [ ]:
print(
    french_to_sql(
        "Afficher tous les étudiants plus de 25 ans"
    )
)

In [ ]:
print(
    french_to_sql(
        "Afficher tous les emloyés plus de 25 ans"
    )
)